In [ ]:
import bp3d
import pandas as pd
import zarr
import numpy as np
import geopandas as gpd
from matplotlib.path import Path
import matplotlib.pyplot as plt

In [ ]:
user = 'user'
password = 'password'

In [ ]:
ensemble_names = ['list', 'of', 'ensemble', 'names']


In [ ]:
burn_unit = gpd.read_file('boundary.geojson')
boundary = gpd.read_file('ff_domain.geojson')


# make the burn unit mask
burn_unit = burn_unit.to_crs(5070)

geometry_string = (burn_unit.geometry[0].wkt)
geometry_string = geometry_string.replace('(', '')
geometry_string = geometry_string.replace(')', '')
geometry_string = geometry_string.replace('MULTIPOLYGON ', '')

geometry_string_list = geometry_string.split(', ')
polygon_vertices = [[float(g.split(' ')[0]), float(g.split(' ')[1])] for g in geometry_string_list]
poly_path = Path(polygon_vertices)

min_x, min_y, max_x, max_y = boundary.to_crs(5070).total_bounds
x_size = round((max_x - min_x) / 2)
y_size = round((max_y - min_y) / 2)

XX = np.linspace(min_x, max_x, x_size)
YY = np.linspace(min_y, max_y, y_size)

# Create a flattened list of all grid coordinates
x_grid_mesh, y_grid_mesh = np.meshgrid(XX, YY)
points = np.vstack((x_grid_mesh.flatten(), y_grid_mesh.flatten())).T

mask_inside = poly_path.contains_points(points)
mask_grid = mask_inside.reshape(x_grid_mesh.shape)


In [ ]:
c = bp3d.Client(user=user, password=password)

surface_consumption = []
midstory_consumption = []
canopy_consumption = []
wind_direction = []
wind_speed = []
lfm = []
fdfm = []
burn_season = []
ignition_scenario = []
viz_link = []

# run this over each ensemble name, then it will loop through the runs in the ensemble
# if there is an issue, it will look at the zarr files to re-calculate the consumption values
# this may take a while if you have a lot that need re-calculating

for n, ensemble_name in enumerate(ensemble_names):
    plan = c.plan(ensemble_name)
    ens = plan.ensemble(ensemble_name)
    runs_df = ens.runs_df
    runs_df = runs_df.reset_index()
    
    for i in np.arange(ens.runs_df.shape[0]):
        
        edf = runs_df.iloc[i]
        rid = edf.run_id
        run_id1 = rid[0:2]
        run_id2 = rid[2:4]
        
        if edf.metrics:
            surface_consumption.append(edf.metrics['inside_surface_consumption'])
            midstory_consumption.append(np.sum(edf.metrics['consumption_t0_tNone_z1_z4']))
            canopy_consumption.append(np.sum(edf.metrics['consumption_t0_tNone_z5_zNone']))
            wind_direction.append(edf.wind_direction)
            wind_speed.append(edf.wind_speed)
            lfm.append(edf.canopy_moisture)
            fdfm.append(edf.surface_moisture)
            burn_season.append(season[n])
            ignition_scenario.append(ignition[n])


            viz_url = edf['viz']
            
            ext = [v.split('.')[3] for v in viz_url]
            
            locations = [i for i, val in enumerate(ext) if val == 'png']
            
            viz_url_subset = []
            for index in locations:
                viz_url_subset.append(viz_url[index])
            
            timestep = [int(v.split('_')[4]) for v in viz_url_subset]
            sorted_url = pd.DataFrame({'url':viz_url_subset, 
                          'time':timestep}).sort_values('time')
            last_viz = sorted_url['url'].values[-1]

            if last_viz.split('_')[-1] == 'legend.png':
                last_viz = (f'{'_'.join(last_viz.split('_')[:-1])}.png')

            viz_link.append(last_viz)


            
        else:

            try:
                z = zarr.open(f'https://wifire-data.sdsc.edu/data/burnpro3d/d/{run_id1}/{run_id2}/run_{rid}/quicfire.zarr', mode="r")
                fuel_density = z['fuels-dens'][...]
                fuel_density_start = fuel_density[0,:,:,:]
                fuel_density_end = fuel_density[-1,:,:,:]
        
                newmask = np.zeros((fuel_density.shape[2],fuel_density.shape[3]))
                newmask[:mask_grid.shape[0], :mask_grid.shape[1]] = ~mask_grid
            
                
                start_surface = np.nansum(np.ma.masked_array(fuel_density_start[0,:,:], newmask))
                end_surface = np.nansum(np.ma.masked_array(fuel_density_end[0,:,:], newmask))
                
                start_mid = np.nansum(np.ma.masked_array(np.sum(fuel_density_start[1:5,:,:], axis = 0), newmask))
                end_mid = np.nansum(np.ma.masked_array(np.sum(fuel_density_end[1:5,:,:], axis = 0), newmask))
                
                start_canopy = np.nansum(np.ma.masked_array(np.sum(fuel_density_start[5:,:,:], axis = 0), newmask))
                end_canopy = np.nansum(np.ma.masked_array(np.sum(fuel_density_end[5:,:,:], axis = 0), newmask))
                
                surface_consumption.append((start_surface - end_surface) / start_surface)
                midstory_consumption.append((start_mid - end_mid) / start_mid)
                canopy_consumption.append((start_canopy - end_canopy) / start_canopy)
                # total_surface_start.append(start_surface)
                # total_midstory_start.append(start_mid)
                # total_canopy_start.append(start_canopy)
                # total_surface_end.append(end_surface)
                # total_midstory_end.append(end_mid)
                # total_canopy_end.append(end_canopy)

                        
            except:
                print(ensemble_name)

In [ ]:
#save the results as a csv file

results_df = pd.DataFrame(dict({
    'season': burn_season,
    'ignition': ignition_scenario,
    'wind_direction':wind_direction,
    'wind_speed':wind_speed,
    'live_fuel_moisture':lfm,
    'fine_dead_fuel_moisture':fdfm,
    'surface_consumption':surface_consumption,
    'midstory_consumption':midstory_consumption,
    'canopy_consumption':canopy_consumption,
    'viz_link': viz_link
    
}))
results_df.to_csv('results.csv')